In [1]:
import os
import asyncio
import json
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceWindowNodeParser, SentenceSplitter
from llama_index.llms.deepseek import DeepSeek
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    BatchEvalRunner,
)
from llama_index.core.evaluation.eval_utils import get_results_df
from llama_index.core.evaluation import DatasetGenerator, QueryResponseDataset

Settings.llm = DeepSeek(model="deepseek-chat", temperature=0.1, api_key=os.getenv("DEEPSEEK_API_KEY"))
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en")


/Users/keen/anaconda3/envs/all-in-rag/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2274: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/keen/anaconda3/envs/all-in-rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
reader = SimpleDirectoryReader(input_files=["../../data/C3/pdf/IPCC_AR6_WGII_Chapter03.pdf"])
documents = reader.load_data()

# 1.1 加载或生成响应评估数据集
if os.path.exists("./c6_response_eval_dataset_1.json"):
    print("加载响应评估数据集...")
    response_eval_dataset = QueryResponseDataset.from_json("./c6_response_eval_dataset_1.json")
else:
    print("生成响应评估数据集...")
    num_questions_per_chunk = 5
    dataset_generator = DatasetGenerator.from_documents(
        documents[:10],  # 减少文档数量
        num_questions_per_chunk=num_questions_per_chunk,
        question_gen_query=(
            f"Generate {num_questions_per_chunk} distinct short-answer questions in English "
            "based solely on the provided context. "
            "Each question must be complete, self-contained, and answerable from the context. "
            "Output exactly one question per line. "
            "Do not include numbering, headings, introductions, summaries, answers, or multiple-choice options."
        ),
    )
    response_eval_dataset = await dataset_generator.agenerate_dataset_from_nodes(num=15)  # 减少问题数量
    response_eval_dataset.save_json("./c6_response_eval_dataset_1.json")



加载响应评估数据集...


/Users/keen/anaconda3/envs/all-in-rag/lib/python3.12/site-packages/llama_index/core/evaluation/dataset_generation.py:110: DeprecationWarning: Call to deprecated class QueryResponseDataset. (Deprecated in favor of `LabelledRagDataset` which should be used instead.)
  return cls(**data)


{'e1bbdacc-be6b-43a3-b14e-4f4f2f7f0ac1': 'Who are the Coordinating Lead Authors of the chapter?',
 '6b3f5b8f-f31a-43f8-a178-c09832dbf579': 'What is the chapter title?',
 '4474b40f-da3e-40d0-ae31-61c795a988be': 'Which report is this chapter part of?',
 '8aa382b1-a64e-445f-a76a-3b5d488c547b': 'What is the DOI of the chapter?',
 'ddf87eaa-a8b2-45ad-931f-3711d8cee74c': 'Who are the Review Editors of the chapter?',
 '7bf2aa33-6150-4cfc-8cd9-d35796520b21': 'What is the title of Chapter 3 of the IPCC AR6 WGII report?',
 'b17da85e-8033-4016-b203-f90660d46420': 'On which page does the Executive Summary of Chapter 3 begin?',
 '52e7f31f-9d97-4af6-bb46-3898136ede8b': 'What is the title of FAQ 3.1?',
 'c755f533-0cc2-451e-8ea0-075b34aa98ae': 'Which section covers observed trends and projections of climatic impact-drivers in the global ocean?',
 '5deeacd7-4787-4355-8a76-09b62ef58c25': 'What is the title of FAQ 3.2?',
 '6454ade5-3430-4233-a365-125e4b2f4dd7': 'What is the title of Box 3.2?',
 'b1b06c9a

In [5]:
queries = response_eval_dataset.questions
queries#.values()

['Who are the Coordinating Lead Authors of the chapter?',
 'What is the chapter title?',
 'Which report is this chapter part of?',
 'What is the DOI of the chapter?',
 'Who are the Review Editors of the chapter?',
 'What is the title of Chapter 3 of the IPCC AR6 WGII report?',
 'On which page does the Executive Summary of Chapter 3 begin?',
 'What is the title of FAQ 3.1?',
 'Which section covers observed trends and projections of climatic impact-drivers in the global ocean?',
 'What is the title of FAQ 3.2?',
 'What is the title of Box 3.2?',
 'What does FAQ 3.3 address regarding the ocean?',
 'What is the topic of Box 3.3?',
 'What does Cross-Chapter Box SLR concern?',
 'What is the subject of FAQ 3.5?']

In [ ]:
def report_evaluation_details(label, queries, results, output_path):
    """打印并保存已有评估结果，不重新调用查询引擎或评估模型。"""
    records = []
    print(f"\n=== {label}：逐题评估明细 ===")
    for i, query in enumerate(queries):
        faith = results["faithfulness"][i]
        relevance = results["relevancy"][i]
        record = {
            "question": query,
            "answer": faith.response,
            "contexts": faith.contexts,
            "evaluations": {
                metric: results[metric][i].model_dump()
                for metric in results
            },
        }
        records.append(record)
        print(f"\n--- 问题 {i + 1}: {query} ---")
        print(f"RAG回答:\n{faith.response}")
        for j, context in enumerate(faith.contexts or [], 1):
            print(f"\n检索上下文 [{j}]:\n{context}")
        for name, evaluation in [("忠实度", faith), ("相关性", relevance)]:
            print(f"\n{name}: passing={evaluation.passing}, score={evaluation.score}")
            print(f"评估模型原始反馈: {evaluation.feedback!r}")
            if evaluation.invalid_result:
                print(f"评估结果无效: {evaluation.invalid_reason}")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"\n明细已保存: {os.path.abspath(output_path)}")


async def main():
    # 1. 加载文档
    reader = SimpleDirectoryReader(input_files=["../../data/C3/pdf/IPCC_AR6_WGII_Chapter03.pdf"])
    documents = reader.load_data()

    # 1.1 加载或生成响应评估数据集
    if os.path.exists("./c6_response_eval_dataset_1.json"):
        print("加载响应评估数据集...")
        response_eval_dataset = QueryResponseDataset.from_json("./c6_response_eval_dataset_1.json")
    else:
        print("生成响应评估数据集...")
        num_questions_per_chunk = 5
        dataset_generator = DatasetGenerator.from_documents(
            documents[:10],  # 减少文档数量
            num_questions_per_chunk=num_questions_per_chunk,
            question_gen_query=(
                f"Generate {num_questions_per_chunk} distinct short-answer questions in English "
                "based solely on the provided context. "
                "Each question must be complete, self-contained, and answerable from the context. "
                "Output exactly one question per line. "
                "Do not include numbering, headings, introductions, summaries, answers, or multiple-choice options."
            ),
        )
        response_eval_dataset = await dataset_generator.agenerate_dataset_from_nodes(num=15)  # 减少问题数量
        response_eval_dataset.save_json("./c6_response_eval_dataset_1.json")



    # 2. 构建两种不同的RAG查询引擎和检索器进行对比
    # 2.1 句子窗口检索
    sentence_parser = SentenceWindowNodeParser.from_defaults(
        window_size=3,
        window_metadata_key="window",
        original_text_metadata_key="original_text",
    )
    sentence_nodes = sentence_parser.get_nodes_from_documents(documents)
    sentence_index = VectorStoreIndex(sentence_nodes)

    sentence_query_engine = sentence_index.as_query_engine(
        similarity_top_k=2,
        node_postprocessors=[
            MetadataReplacementPostProcessor(target_metadata_key="window")
        ],
    )
    sentence_retriever = sentence_index.as_retriever(similarity_top_k=2)

    # 2.2 常规分块检索（基准）
    base_parser = SentenceSplitter(chunk_size=512)
    base_nodes = base_parser.get_nodes_from_documents(documents)
    base_index = VectorStoreIndex(base_nodes)

    base_query_engine = base_index.as_query_engine(similarity_top_k=2)
    base_retriever = base_index.as_retriever(similarity_top_k=2)

    # 3. 初始化响应评估器
    faithfulness_evaluator = FaithfulnessEvaluator(llm=Settings.llm)
    relevancy_evaluator = RelevancyEvaluator(llm=Settings.llm)

    # 4. 执行响应评估对比
    print("开始执行响应评估对比...")
    evaluators = {"faithfulness": faithfulness_evaluator, "relevancy": relevancy_evaluator}
    # 保存实际使用的评估提示词，方便核对判分标准。
    with open("c6_evaluation_prompts.json", "w", encoding="utf-8") as f:
        json.dump(
            {name: {key: prompt.get_template() for key, prompt in evaluator.get_prompts().items()}
             for name, evaluator in evaluators.items()},
            f, ensure_ascii=False, indent=2,
        )
    queries = response_eval_dataset.queries

    # 句子窗口检索响应评估
    print("\n=== 评估句子窗口检索 ===")
    sentence_runner = BatchEvalRunner(evaluators, workers=2, show_progress=True)
    sentence_response_results = await sentence_runner.aevaluate_queries(
        queries=queries, query_engine=sentence_query_engine
    )
    report_evaluation_details(
        "句子窗口检索", queries, sentence_response_results, "c6_sentence_eval_details.json"
    )

    # 常规分块检索响应评估
    print("\n=== 评估常规分块检索 ===")
    base_runner = BatchEvalRunner(evaluators, workers=2, show_progress=True)
    base_response_results = await base_runner.aevaluate_queries(
        queries=queries, query_engine=base_query_engine
    )
    report_evaluation_details(
        "常规分块检索", queries, base_response_results, "c6_base_eval_details.json"
    )

    # 5. 分析并打印对比结果
    print("\n" + "="*60)
    print("响应评估结果对比")
    print("="*60)

    def calc_response_score(results, metric):
        if results and results.get(metric):
            scores = results[metric]
            return sum(r.passing for r in scores) / len(scores)
        return 0

    # 句子窗口检索结果
    sentence_faith = calc_response_score(sentence_response_results, "faithfulness")
    sentence_rel = calc_response_score(sentence_response_results, "relevancy")

    # 常规分块检索结果
    base_faith = calc_response_score(base_response_results, "faithfulness")
    base_rel = calc_response_score(base_response_results, "relevancy")

    print(f"\n句子窗口检索:")
    print(f"  忠实度: {sentence_faith:.1%}")
    print(f"  相关性: {sentence_rel:.1%}")

    print(f"\n常规分块检索:")
    print(f"  忠实度: {base_faith:.1%}")
    print(f"  相关性: {base_rel:.1%}")



    # 简单对比
    if sentence_faith > base_faith and sentence_rel > base_rel:
        print(f"\n✅ 句子窗口检索在两个维度上都优于常规分块检索")
    elif sentence_faith > base_faith or sentence_rel > base_rel:
        print(f"\n⚖️  句子窗口检索在某些维度上有优势")
    else:
        print(f"\n❌ 句子窗口检索未显示明显优势")



if __name__ == "__main__":
    asyncio.run(main())
